<a href="https://colab.research.google.com/github/pradervonsky/vbig-lab/blob/main/evaluation/generation-9_InternVL2-1B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SVLM Dashboard Insight Generation

## Initial Steps

In [1]:
!pip install -q supabase pillow requests torch torchvision einops

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 kB 3.4 MB/s eta 0:00:00


In [2]:
import os
import time
import requests
import torch
from io import BytesIO
from PIL import Image
from supabase import create_client, Client
from google.colab import userdata
from huggingface_hub import login

In [3]:
# Supabase credentials
SUPABASE_URL = userdata.get("SUPABASE_URL")
SUPABASE_KEY = userdata.get("SUPABASE_KEY")
HF_TOKEN     = userdata.get("HF_TOKEN")

supabase: Client = create_client(SUPABASE_URL, SUPABASE_KEY)
print("Supabase client initialised.")

login(token=HF_TOKEN)
print("HuggingFace login successful.")

Supabase client initialised.
HuggingFace login successful.


In [4]:
# Pull qualifying metadata_ids from human_insights
hi_response = supabase.table("human_insights") \
    .select("metadata_id") \
    .eq("expected_dataset", True) \
    .is_("rejection_reason", "null") \
    .execute()

qualified_ids = list({row["metadata_id"] for row in hi_response.data})
print(f"Qualified dashboards: {len(qualified_ids)}")

# Pull metadata only for those ids
response = supabase.table("metadata") \
    .select("id, bucket_path") \
    .in_("id", qualified_ids) \
    .execute()
dashboards = response.data

print(f"Loaded {len(dashboards)} dashboards.")
print("Sample record:", dashboards[0] if dashboards else "(empty)")

Qualified dashboards: 40
Loaded 40 dashboards.
Sample record: {'id': 'abee2e83-6384-4c23-abfd-e5ede8b5a7bf', 'bucket_path': 'screenshots/abee2e83-6384-4c23-abfd-e5ede8b5a7bf.png'}


In [5]:
# Build public image URL from bucket_path
def build_image_url(bucket_path: str) -> str:
    return f"{SUPABASE_URL}/storage/v1/object/public/superstore/{bucket_path}"

# Fetch image from URL and return a PIL Image
def fetch_image(url: str) -> Image.Image:
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    return Image.open(BytesIO(resp.content)).convert("RGB")

# Resize the image prior the inference pipeline
def prepare_image(image: Image.Image, max_width=2000, max_height=1500) -> Image.Image:
    if image.width > max_width or image.height > max_height:
        ratio = min(max_width / image.width, max_height / image.height)
        new_size = (int(image.width * ratio), int(image.height * ratio))
        image = image.resize(new_size)
        print(f"  Resized to {new_size}")
    return image

# Extract model identity from a loaded model object
def get_model_meta(model, hf_id=None):
    cfg   = getattr(model, "config", None)
    hf_id = hf_id or getattr(cfg, "_name_or_path", None)
    name  = hf_id.split("/")[-1] if hf_id else None
    return {"model_name": name, "model_hf_id": hf_id}

In [6]:
# Prompt
PROMPT = """
You are a senior BI analyst generating structured insights for a business intelligence dashboard.
Think step-by-step: first extract visible quantitative facts, then identify visual patterns, then derive business implications.

Before writing any chart analysis, count the number of distinct charts visible in the dashboard and write: 'Chart count: N'.
Then produce exactly N chart analyses and no more.
Analyze chart-by-chart in Z-pattern (left to right, top to bottom).
If there are scoreboard/scorecard charts (e.g., sales, profit, orders, customers, etc), treat them as the first chart as one single chart with the title of "Scoreboard Overview".
Once grouped into Scoreboard Overview, those KPI panels are fully analyzed and must never appear again as individual charts anywhere in your output.
If there are no scoreboard/scorecard charts, proceed with writing the first chart available.
Do not treat UI labels, navigation tabs, filters, or sidebar controls as charts.

For each chart, write exactly:
L2: one sentence reporting only values explicitly shown or labeled: highest/lowest value, comparison, ranking, or proportion only. Do not compute anything not displayed in the image.
L3: one sentence describing a visual pattern: a direction, a shape, a gap, or an exception. Use natural language: "volatile", "dipped", "wider margin", "considerably far", "spread". Use hedging: "appears to", "seems to", "suggesting". Write NOT APPLICABLE if the chart is: a ranked table, a top-N list, or a gauge.
L4: one sentence connecting the pattern to business context or domain knowledge not visible in the chart. Must reference a specific value from L2 or a specific pattern from L3; never use generic phrases such as 'this could be due to' without grounding them in what was observed. Never restate what is already visible. Always required.

Output format:
Chart 1: [Title]
L2: [One sentence.]
L3: [One sentence.] or NOT APPLICABLE
L4: [One sentence.]

Chart 2: [Title]
L2: [One sentence.]
L3: [One sentence.] or NOT APPLICABLE
L4: [One sentence.]

Rules:
- Write EXACTLY 1 sentence per level per chart
- Skip navigation tabs, filters, sidebar controls, and dropdowns entierly
- Do not include axis labels, colors, or chart type names
- Immediately after writing your final chart analysis, write END OF ANALYSIS on its own line and generate no further text under any circumstances
"""

In [7]:
# Quick sanity check on the first dashboard
if dashboards:
    sample_url = build_image_url(dashboards[0]["bucket_path"])
    print("Sample URL:", sample_url)
    sample_img = fetch_image(sample_url)
    print("Image size:", sample_img.size)
    sample_img

Sample URL: https://olduvnqhykovcfbfouhe.supabase.co/storage/v1/object/public/superstore/screenshots/abee2e83-6384-4c23-abfd-e5ede8b5a7bf.png
Image size: (1200, 927)


---

## InternVL2-1B
https://huggingface.co/OpenGVLab/InternVL2-1B  

In [8]:
import subprocess
subprocess.run(["pip", "install", "-q", "transformers==4.37.2"], check=True)

import importlib
import transformers
importlib.reload(transformers)

from transformers import AutoModelForCausalLM, AutoModel, AutoTokenizer
print(transformers.__version__)

4.37.2


In [9]:
MODEL_HF_ID    = "OpenGVLab/InternVL2-1B"

device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_HF_ID,
    trust_remote_code=True,
    token=HF_TOKEN,
)
model = AutoModel.from_pretrained(
    MODEL_HF_ID,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    use_flash_attn=False,
    trust_remote_code=True,
    token=HF_TOKEN,
).eval().to(device)

meta = get_model_meta(model, hf_id=MODEL_HF_ID)
print(f"Loaded on {device}:")
print(f"  model_name:     {meta['model_name']}")
print(f"  model_hf_id:    {meta['model_hf_id']}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `hf_hub_download`. Downloads always resume whenever possible.
  is thrown and the `use_auth_token` value is ignored.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/265 [00:00<?, ?B/s]

...

A new version of the following files was downloaded from https://huggingface.co/OpenGVLab/InternVL2-1B:
- conversation.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/OpenGVLab/InternVL2-1B:
- modeling_internvl_chat.py
- modeling_intern_vit.py
- conversation.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:189: UserWarning: The `resume_download` argument is

FlashAttention2 is not installed.


model.safetensors:   0%|          | 0.00/1.88G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/117 [00:00<?, ?B/s]

Loaded on cuda:
  model_name:     InternVL2-1B
  model_hf_id:    OpenGVLab/InternVL2-1B


In [10]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

def build_transform(input_size):
    return T.Compose([
        T.Lambda(lambda img: img.convert('RGB') if img.mode != 'RGB' else img),
        T.Resize((input_size, input_size), interpolation=InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])

def find_closest_aspect_ratio(aspect_ratio, target_ratios, width, height, image_size):
    best_ratio_diff = float('inf')
    best_ratio = (1, 1)
    area = width * height
    for ratio in target_ratios:
        target_aspect_ratio = ratio[0] / ratio[1]
        ratio_diff = abs(aspect_ratio - target_aspect_ratio)
        if ratio_diff < best_ratio_diff:
            best_ratio_diff = ratio_diff
            best_ratio = ratio
        elif ratio_diff == best_ratio_diff:
            if area > 0.5 * image_size * image_size * ratio[0] * ratio[1]:
                best_ratio = ratio
    return best_ratio

def dynamic_preprocess(image, min_num=1, max_num=12, image_size=448, use_thumbnail=False):
    orig_width, orig_height = image.size
    aspect_ratio = orig_width / orig_height
    target_ratios = set(
        (i, j) for n in range(min_num, max_num + 1)
        for i in range(1, n + 1) for j in range(1, n + 1)
        if i * j <= max_num and i * j >= min_num
    )
    target_ratios = sorted(target_ratios, key=lambda x: x[0] * x[1])
    target_aspect_ratio = find_closest_aspect_ratio(
        aspect_ratio, target_ratios, orig_width, orig_height, image_size)
    target_width  = image_size * target_aspect_ratio[0]
    target_height = image_size * target_aspect_ratio[1]
    blocks = target_aspect_ratio[0] * target_aspect_ratio[1]
    resized_img = image.resize((target_width, target_height))
    processed_images = []
    for i in range(blocks):
        box = (
            (i % (target_width // image_size)) * image_size,
            (i // (target_width // image_size)) * image_size,
            ((i % (target_width // image_size)) + 1) * image_size,
            ((i // (target_width // image_size)) + 1) * image_size,
        )
        processed_images.append(resized_img.crop(box))
    if use_thumbnail and len(processed_images) != 1:
        processed_images.append(image.resize((image_size, image_size)))
    return processed_images

def load_internvl_image(image: Image.Image, input_size=448, max_num=12):
    transform = build_transform(input_size)
    images = dynamic_preprocess(image, image_size=input_size, use_thumbnail=True, max_num=max_num)
    pixel_values = [transform(img) for img in images]
    return torch.stack(pixel_values).to(torch.bfloat16).to(device)

### Testing one sample generation

In [11]:
# import torchvision.transforms as T
# from torchvision.transforms.functional import InterpolationMode

# # single-image single-round, question must be prefixed with <image>\n
# test_dashboard = dashboards[0]
# test_image     = fetch_image(build_image_url(test_dashboard["bucket_path"]))

# question      = f"<image>\n{PROMPT}"
# pixel_values  = load_internvl_image(test_image)
# generation_config = dict(max_new_tokens=1024, do_sample=False)

# t0          = time.perf_counter()
# test_output = model.chat(tokenizer, pixel_values, question, generation_config)
# test_ms     = int((time.perf_counter() - t0) * 1000)

# print(f"Dashboard ID:   {test_dashboard['id']}")
# print(f"Inference time: {test_ms} ms")
# print(f"\nOutput:\n{test_output}")
# display(test_image)

# supabase.table("vlm_outputs").upsert({
#     "metadata_id":       test_dashboard["id"],
#     **meta,
#     "raw_output":        test_output,
#     "inference_success": True,
#     "error_message":     None,
#     "inference_ms":      test_ms,
# }, on_conflict="metadata_id,model_name").execute()

# print("Saved to vlm_outputs.")

### 40 dashboards generation

In [12]:
import torchvision.transforms as T
from torchvision.transforms.functional import InterpolationMode
from tqdm import tqdm

question = f"<image>\n{PROMPT}"
generation_config = dict(max_new_tokens=1024, do_sample=False)

ok  = 0
err = 0

for dashboard in tqdm(dashboards, desc="Generating", unit="dashboard"):
    dashboard_id = dashboard["id"]

    try:
        image        = prepare_image(fetch_image(build_image_url(dashboard["bucket_path"])))
        pixel_values = load_internvl_image(image).to(torch.bfloat16).to(device)

        t0     = time.perf_counter()
        output = model.chat(tokenizer, pixel_values, question, generation_config)
        elapsed_ms = int((time.perf_counter() - t0) * 1000)

        supabase.table("vlm_outputs").upsert({
            "metadata_id":       dashboard_id,
            **meta,
            "raw_output":        output,
            "inference_success": True,
            "error_message":     None,
            "inference_ms":      elapsed_ms,
        }, on_conflict="metadata_id,model_name").execute()

        ok += 1
        print(f"[OK]  {dashboard_id}  ({elapsed_ms} ms)")
        print(f"      {output[:120]}...\n")

    except Exception as e:
        supabase.table("vlm_outputs").upsert({
            "metadata_id":       dashboard_id,
            **meta,
            "raw_output":        None,
            "inference_success": False,
            "error_message":     str(e),
            "inference_ms":      None,
        }, on_conflict="metadata_id,model_name").execute()

        err += 1
        print(f"[ERR] {dashboard_id}: {e}")

print(f"\nDone. {ok} succeeded, {err} failed.")

Generating:   2%|▎         | 1/40 [00:49<32:01, 49.26s/dashboard]

[OK]  abee2e83-6384-4c23-abfd-e5ede8b5a7bf  (48156 ms)
      Chart 1: Superstore Sales Overview
L2: Sales Target
L3: Sales by Sub-Category
L4: Sales by State

Chart 2: Superstore Sa...



Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Generating:   5%|▌         | 2/40 [01:30<28:15, 44.63s/dashboard]

[OK]  ee87f028-0bf2-4c03-81c5-6b974a4cfcb5  (40419 ms)
      Chart 1: Welcome back, (<Full Name>)
L2: Sales
L3: Sales last month: +198.1%, Profit last month: +830.8%, Total Order: 2...



Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Generating:   8%|▊         | 3/40 [02:16<27:56, 45.30s/dashboard]

...

[OK]  1c6fba5d-67a4-4f86-b7a8-6f5f37c4dd30  (40963 ms)
      Chart 1: Profit Ratio Trend
L2: Profit Ratio Trend appears to be a fluctuating line with peaks and troughs, indicating a...



Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Generating:  98%|█████████▊| 39/40 [26:06<00:38, 38.78s/dashboard]

[OK]  aa95ffcc-e6f4-4869-9784-92bd011b3b79  (31724 ms)
      Chart 1: Product Performance Dashboard
L2: 
- Highest sales in January: 14,966
- Highest sales in February: 14,561
- Hig...



Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Generating: 100%|██████████| 40/40 [26:53<00:00, 40.33s/dashboard]

[OK]  932c11c0-d78c-4651-8bb9-73325937333d  (45659 ms)
      Chart 1: Overview Dashboard
L2: Sales by State
L3: Sales by Segment
L4: Sales by Manufacturer

Chart 2: Overview Dashboa...


Done. 40 succeeded, 0 failed.
